<a href="https://colab.research.google.com/github/tobiartinian/Econom-a-y-Finanzas/blob/main/Features_Enginering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
%pip install jupysql
%pip install duckdb-engine
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb://

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.0/879.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 749.0 kB/s eta 0:00:00


In [ ]:
dataset_filename = "competencia_01.csv"
dataset_path = "/content/drive/MyDrive/Economía y finanzas"

In [ ]:
semilla = 772859

# Feature Enginering

## Generamos nuevas variables LAG



In [ ]:
df = pd.read_csv(f'{dataset_path}/{dataset_filename}')

# Columnas que deseas ignorar
excluir = {'foto_mes', 'numero_de_cliente', 'clase_ternaria'}

# Obtener todas las columnas excepto las 3 especificadas
variables_lag = [col for col in df.columns if col not in excluir]

In [ ]:
file_path = f"{dataset_path}/{dataset_filename}"
con = duckdb.connect("competencia.duckdb")

# 1. Leer solo las columnas del CSV usando Pandas (nrows=0 lee instantáneamente)
df_header = pd.read_csv(file_path, nrows=0)
columnas_existentes = set(df_header.columns)

# 2. Excluir las variables que NO deben llevar Lag
excluir = {'foto_mes', 'numero_de_cliente', 'clase_ternaria'}
variables_lag_validas = [col for col in df_header.columns if col not in excluir]

# 3. Validar si alguna variable esperada faltó (Opcional)
if 'variables_lag' in locals():
    columnas_faltantes = set(variables_lag) - columnas_existentes
    if columnas_faltantes:
        print(f"⚠️ Las siguientes columnas no existen y fueron ignoradas: {columnas_faltantes}")

# 4. Generar la consulta SQL para los Lags
lags_sql = ",\n    ".join([
    f"LAG({col}, 1) OVER (PARTITION BY numero_de_cliente ORDER BY foto_mes) AS {col}_lag1"
    for col in variables_lag_validas
])

# 5. Ejecutar la creación de la tabla en DuckDB
query = f"""
CREATE OR REPLACE TABLE competencia_01 AS
SELECT
    *,
    {lags_sql}
FROM read_csv_auto('{file_path}');
"""

con.execute(query)
print(f"¡Tabla cargada con éxito! Se procesaron {len(variables_lag_validas)} variables con Lag.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

¡Tabla cargada con éxito! Se procesaron 152 variables con Lag.


# Modelo

## Entrenamiento del modelo

In [ ]:
# 1. Conectar a DuckDB
con = duckdb.connect("competencia.duckdb")

# 2. Descubrir las features excluyendo las columnas que no van
df_header = con.execute("SELECT * FROM competencia_01 LIMIT 0").df()
excluir = {"foto_mes", "numero_de_cliente", "clase_ternaria"}
features = [col for col in df_header.columns if col not in excluir]
features_sql = ", ".join(features)

# 3. Traer a Pandas ÚNICAMENTE el rango de meses necesario (ej. Marzo a Junio 2021)
print("Cargando datos filtrados desde DuckDB...")
query = f"""
    SELECT
        {features_sql},
        CASE WHEN clase_ternaria = 'BAJA+2' THEN 1 ELSE 0 END AS target
    FROM competencia_01
    WHERE foto_mes BETWEEN 202103 AND 202106
"""
df_filtrado = con.execute(query).df()


Cargando datos filtrados desde DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
df_filtrado.to_csv('/content/drive/MyDrive/Economía y finanzas/df_filtrado.csv', index=False)